### Import data

##### DuckDB is used because M5 dataset is large, and converting it from wide to long format is a memory-intensive operation. 
##### DuckDB allows us to perform the transformation through SQL and write the result directly to Parquet. This avoids requiring the entire resulting DataFrame to be held in pandas memory.
##### **DuckDB is not automatically faster or more memory-efficient in every situation. Its advantage here is the ability to execute SQL transformations and write results directly to disk.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd


In [2]:
# Locate project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SALES_FILE = RAW_DIR / "sales_train_evaluation.csv"
OUTPUT_FILE = PROCESSED_DIR / "daily_sales.parquet"

print("Project root:", PROJECT_ROOT)
print("Sales file exists:", SALES_FILE.exists())
print("Output directory:", PROCESSED_DIR)

Project root: c:\Users\cjc72\Desktop\DeliveryLate\AI-Retail-Demand-Intelligence
Sales file exists: True
Output directory: c:\Users\cjc72\Desktop\DeliveryLate\AI-Retail-Demand-Intelligence\data\processed


In [3]:
# Inspect DuckDB environment
con = duckdb.connect()

print("DuckDB version:", duckdb.__version__)

con.execute("SELECT 42 AS test").df()

DuckDB version: 2.0.0


,test
0,42


### Convert sales into long format

In [4]:
metadata_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

excluded_columns = ", ".join(metadata_columns)

query = f"""
COPY (
    SELECT
        id,
        item_id,
        dept_id,
        cat_id,
        store_id,
        state_id,
        d,
        sales
    FROM read_csv_auto('{SALES_FILE.as_posix()}', header = true)
    UNPIVOT (
        sales FOR d IN (
            COLUMNS(* EXCLUDE ({excluded_columns}))
        )
    )
)
TO '{OUTPUT_FILE.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(query)

print("Daily sales dataset created:", OUTPUT_FILE)

Daily sales dataset created: c:\Users\cjc72\Desktop\DeliveryLate\AI-Retail-Demand-Intelligence\data\processed\daily_sales.parquet


In [5]:
# Validate the trasnformed dataset
daily_sales = con.execute(f"""
    SELECT *
    FROM read_parquet('{OUTPUT_FILE.as_posix()}')
    LIMIT 10
""").df()

daily_sales

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_2,0
2,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_3,0
3,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_4,0
4,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_5,0
5,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_6,0
6,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_7,0
7,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_8,0
8,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_9,0
9,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_10,0


In [6]:
# Check total number of rows
row_count = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{OUTPUT_FILE.as_posix()}')
""").fetchone()[0]

print(f"Total observations: {row_count:,}")

Total observations: 59,181,090


#### Check number of unique series and days

In [7]:
validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(DISTINCT d) AS unique_days,
        MIN(sales) AS minimum_sales,
        MAX(sales) AS maximum_sales,
        SUM(CASE WHEN sales IS NULL THEN 1 ELSE 0 END) AS null_sales
    FROM read_parquet('{OUTPUT_FILE.as_posix()}')
""").df()

validation

,total_rows,unique_series,unique_days,minimum_sales,maximum_sales,null_sales
0,59181090,30490,1941,0,763,0.0


#### Check for duplicates

In [8]:
duplicate_check = con.execute(f"""
    SELECT COUNT(*) AS duplicate_groups
    FROM (
        SELECT id, d
        FROM read_parquet('{OUTPUT_FILE.as_posix()}')
        GROUP BY id, d
        HAVING COUNT(*) > 1
    )
""").df()

duplicate_check

,duplicate_groups
0,0


### Join Calendar data

In [9]:
# Load the calendar dataset
CALENDAR_FILE = RAW_DIR / "calendar.csv"

calendar = pd.read_csv(CALENDAR_FILE)

calendar["date"] = pd.to_datetime(calendar["date"])

print("Calendar shape:", calendar.shape)

calendar.head()

Calendar shape: (1969, 14)


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [10]:
print("Total calendar rows:", len(calendar))
print("Unique day identifiers:", calendar["d"].nunique())
print("Duplicate day identifiers:", calendar["d"].duplicated().sum())

print("Missing values:")
display(calendar.isnull().sum())

Total calendar rows: 1969
Unique day identifiers: 1969
Duplicate day identifiers: 0
Missing values:


date               0
wm_yr_wk           0
weekday            0
wday               0
month              0
year               0
d                  0
event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
snap_CA            0
snap_TX            0
snap_WI            0
dtype: int64

In [11]:
# Perform join between daily sales and calendar datasets
CALENDAR_OUTPUT = PROCESSED_DIR / "daily_sales_calendar.parquet"

calendar_columns = [
    "d",
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

calendar_sql = ", ".join(calendar_columns)

query = f"""
COPY (
    SELECT
        s.*,
        c.date,
        c.wm_yr_wk,
        c.weekday,
        c.wday,
        c.month,
        c.year,
        c.event_name_1,
        c.event_type_1,
        c.event_name_2,
        c.event_type_2,
        c.snap_CA,
        c.snap_TX,
        c.snap_WI
    FROM read_parquet('{OUTPUT_FILE.as_posix()}') AS s
    LEFT JOIN read_csv_auto('{CALENDAR_FILE.as_posix()}') AS c
        ON s.d = c.d
)
TO '{CALENDAR_OUTPUT.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(query)

print("Calendar-enriched dataset created:", CALENDAR_OUTPUT)


Calendar-enriched dataset created: c:\Users\cjc72\Desktop\DeliveryLate\AI-Retail-Demand-Intelligence\data\processed\daily_sales_calendar.parquet


In [12]:
calendar_validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(DISTINCT d) AS unique_days,
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END)
            AS missing_dates,
        SUM(CASE WHEN wm_yr_wk IS NULL THEN 1 ELSE 0 END)
            AS missing_retail_weeks,
        SUM(CASE WHEN snap_CA IS NULL THEN 1 ELSE 0 END)
            AS missing_snap_ca,
        SUM(CASE WHEN snap_TX IS NULL THEN 1 ELSE 0 END)
            AS missing_snap_tx,
        SUM(CASE WHEN snap_WI IS NULL THEN 1 ELSE 0 END)
            AS missing_snap_wi
    FROM read_parquet('{CALENDAR_OUTPUT.as_posix()}')
""").df()

calendar_validation

,total_rows,unique_series,unique_days,missing_dates,missing_retail_weeks,missing_snap_ca,missing_snap_tx,missing_snap_wi
0,59181090,30490,1941,0.0,0.0,0.0,0.0,0.0


### Inspect the joined dataset

In [13]:
sample = con.execute(f"""
    SELECT *
    FROM read_parquet('{CALENDAR_OUTPUT.as_posix()}')
    ORDER BY id, date
    LIMIT 10
""").df()

sample

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,1,1,2011,NaN,NaN,None,None,0,0,0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,2,1,2011,NaN,NaN,None,None,0,0,0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,3,1,2011,NaN,NaN,None,None,0,0,0
3,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,4,2,2011,NaN,NaN,None,None,1,1,0
4,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,5,2,2011,NaN,NaN,None,None,1,0,1
5,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_6,2,2011-02-03,11101,...,6,2,2011,NaN,NaN,None,None,1,1,1
6,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_7,0,2011-02-04,11101,...,7,2,2011,NaN,NaN,None,None,1,0,0
7,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_8,2,2011-02-05,11102,...,1,2,2011,NaN,NaN,None,None,1,1,1
8,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_9,0,2011-02-06,11102,...,2,2,2011,SuperBowl,Sporting,None,None,1,1,1
9,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_10,0,2011-02-07,11102,...,3,2,2011,NaN,NaN,None,None,1,1,0


In [14]:
calendar_validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(DISTINCT d) AS unique_days,
        COUNT(DISTINCT date) AS unique_dates,
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS missing_dates,
        SUM(CASE WHEN wm_yr_wk IS NULL THEN 1 ELSE 0 END) AS missing_retail_weeks,
        SUM(CASE WHEN snap_CA IS NULL THEN 1 ELSE 0 END) AS missing_snap_ca,
        SUM(CASE WHEN snap_TX IS NULL THEN 1 ELSE 0 END) AS missing_snap_tx,
        SUM(CASE WHEN snap_WI IS NULL THEN 1 ELSE 0 END) AS missing_snap_wi
    FROM read_parquet('{CALENDAR_OUTPUT.as_posix()}')
""").df()

display(calendar_validation.T)

,0
total_rows,59181090.0
unique_series,30490.0
unique_days,1941.0
unique_dates,1941.0
missing_dates,0.0
missing_retail_weeks,0.0
missing_snap_ca,0.0
missing_snap_tx,0.0
missing_snap_wi,0.0


### Integrate weekly selling prices

In [15]:
# Load and inspect the selling price data
PRICE_FILE = RAW_DIR / "sell_prices.csv"

prices = pd.read_csv(PRICE_FILE)

print("Selling prices shape:", prices.shape)
display(prices.head())

print("Total rows:", len(prices))
print("Unique stores:", prices["store_id"].nunique())
print("Unique items:", prices["item_id"].nunique())
print("Unique retail weeks:", prices["wm_yr_wk"].nunique())

print("\nMissing values:")
display(prices.isnull().sum())

print("\nDuplicate key combinations:")
print(
    prices.duplicated(
        subset=["store_id", "item_id", "wm_yr_wk"]
    ).sum()
)

print("\nPrice statistics:")
display(prices["sell_price"].describe())

Selling prices shape: (6841121, 4)


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


Total rows: 6841121
Unique stores: 10
Unique items: 3049
Unique retail weeks: 282

Missing values:


store_id      0
item_id       0
wm_yr_wk      0
sell_price    0
dtype: int64


Duplicate key combinations:
0

Price statistics:


count    6.841121e+06
mean     4.410952e+00
std      3.408814e+00
min      1.000000e-02
25%      2.180000e+00
50%      3.470000e+00
75%      5.840000e+00
max      1.073200e+02
Name: sell_price, dtype: float64

#### Join selling prices using DuckDB

In [16]:
PRICE_OUTPUT = PROCESSED_DIR / "daily_sales_calendar_prices.parquet"

query = f"""
COPY (
    SELECT
        s.*,
        p.sell_price
    FROM read_parquet('{CALENDAR_OUTPUT.as_posix()}') AS s
    LEFT JOIN read_csv_auto('{PRICE_FILE.as_posix()}') AS p
        ON s.store_id = p.store_id
        AND s.item_id = p.item_id
        AND s.wm_yr_wk = p.wm_yr_wk
)
TO '{PRICE_OUTPUT.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(query)

print("Saved:", PRICE_OUTPUT)

# Use LEFT JOIN to retain every sales observation, including observations without a matching selling price. 
# An inner join would silently discard those records and distort the historical sales panel.

Saved: c:\Users\cjc72\Desktop\DeliveryLate\AI-Retail-Demand-Intelligence\data\processed\daily_sales_calendar_prices.parquet


#### Validate

In [17]:
price_validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(DISTINCT d) AS unique_days,
        COUNT(DISTINCT date) AS unique_dates,

        SUM(
            CASE WHEN sell_price IS NULL THEN 1 ELSE 0 END
        ) AS missing_prices,

        SUM(
            CASE WHEN sell_price <= 0 THEN 1 ELSE 0 END
        ) AS non_positive_prices,

        MIN(sell_price) AS minimum_price,
        MAX(sell_price) AS maximum_price
    FROM read_parquet('{PRICE_OUTPUT.as_posix()}')
""").df()

display(price_validation.T)

,0
total_rows,59181090.00
unique_series,30490.00
unique_days,1941.00
unique_dates,1941.00
missing_prices,12299413.00
non_positive_prices,0.00
minimum_price,0.01
maximum_price,107.32


### Detect missing price

In [18]:
missing_price_analysis = con.execute(f"""
    SELECT
        COUNT(*) AS total_observations,

        SUM(
            CASE WHEN sell_price IS NULL THEN 1 ELSE 0 END
        ) AS missing_price_observations,

        ROUND(
            100.0 * SUM(
                CASE WHEN sell_price IS NULL THEN 1 ELSE 0 END
            ) / COUNT(*),
            2
        ) AS missing_price_percentage,

        COUNT(DISTINCT CASE
            WHEN sell_price IS NULL THEN id
        END) AS series_with_missing_prices

    FROM read_parquet('{PRICE_OUTPUT.as_posix()}')
""").df()

display(missing_price_analysis)

,total_observations,missing_price_observations,missing_price_percentage,series_with_missing_prices
0,59181090,12299413.0,20.78,19558


#### Check if missing price correspond to zero-sales observations

In [19]:
missing_price_sales = con.execute(f"""
    SELECT
        CASE
            WHEN sales = 0 THEN 'Zero sales'
            ELSE 'Positive sales'
        END AS sales_status,

        COUNT(*) AS observations

    FROM read_parquet('{PRICE_OUTPUT.as_posix()}')
    WHERE sell_price IS NULL
    GROUP BY sales_status
""").df()

display(missing_price_sales)

,sales_status,observations
0,Zero sales,12299413


### Final Checking

In [20]:
final_panel_audit = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(DISTINCT item_id) AS unique_items,
        COUNT(DISTINCT store_id) AS unique_stores,
        COUNT(DISTINCT d) AS unique_days,
        COUNT(DISTINCT date) AS unique_dates,

        MIN(date) AS first_date,
        MAX(date) AS last_date,

        SUM(CASE WHEN sales IS NULL THEN 1 ELSE 0 END)
            AS missing_sales,

        SUM(CASE WHEN sales < 0 THEN 1 ELSE 0 END)
            AS negative_sales,

        SUM(CASE WHEN sell_price IS NULL THEN 1 ELSE 0 END)
            AS missing_prices

    FROM read_parquet('{PRICE_OUTPUT.as_posix()}')
""").df()

display(final_panel_audit.T)

,0
total_rows,59181090
unique_series,30490
unique_items,3049
unique_stores,10
unique_days,1941
unique_dates,1941
first_date,2011-01-29 00:00:00
last_date,2016-05-22 00:00:00
missing_sales,0.0
negative_sales,0.0


## Chronological Split Strategy

- **Multiple folds:** Evaluate model performance across different forecast origins to reduce dependence on a single validation period.
- **Three development folds:** A practical compromise between evaluation reliability and computational cost. However, the 3 consecutive folds cover only 84 days, so they do not establish broad seasonal robustness.
- **28-day forecast horizon:** Aligns with the M5 competition and represents a four-week retail planning period.
- **Expanding-window validation:** Retains all historical observations while progressively adding newer data, simulating periodic model retraining.
- **Initial training period (d_1–d_1801):** Preserves approximately 4.93 years of historical data while reserving 140 days for development validation, internal testing, and M5 validation.

### Data Split Configuration

| Dataset | Training Period | Evaluation Period |
|---|---|---|
| Fold 1 | d_1–d_1801 | d_1802–d_1829 |
| Fold 2 | d_1–d_1829 | d_1830–d_1857 |
| Fold 3 | d_1–d_1857 | d_1858–d_1885 |
| Internal Test | d_1–d_1885 | d_1886–d_1913 |
| M5 Validation | d_1–d_1913 | d_1914–d_1941 |

**Note:** d_1914–d_1941 is the M5 validation period, not the hidden evaluation period. The hidden evaluation period is d_1942–d_1969, for which actual sales are not provided in the dataset.

**Decision:** Retain three development folds for initial experiments. Consider a fourth fold if computational resources permit, and apply the same revised splits to all models for fair comparison.

In [21]:
split_config = {
    "forecast_horizon": 28,

    "development_folds": [
        {
            "fold": 1,
            "train_end": 1801,          # Training: d_1–d_1801
            "validation_start": 1802,
            "validation_end": 1829      # Validation: d_1802–d_1829
        },
        {
            "fold": 2,
            "train_end": 1829,          # Training: d_1–d_1829
            "validation_start": 1830,
            "validation_end": 1857      # Validation: d_1830–d_1857
        },
        {
            "fold": 3,
            "train_end": 1857,          # Training: d_1–d_1857
            "validation_start": 1858,
            "validation_end": 1885      # Validation: d_1858–d_1885
        }
    ],

    "internal_test": {
        "train_end": 1885,    
        "test_start": 1886,
        "test_end": 1913              # d_1886–d_1913
    },

    "official_evaluation": {
        "train_end": 1913,
        "evaluation_start": 1914,
        "evaluation_end": 1941        # d_1914–d_1941
    }
}

### Save this configuration

In [22]:
import json

METADATA_DIR = Path("../data/metadata")

with open(
    METADATA_DIR / "forecast_split_config.json",
    "w"
) as f:
    json.dump(split_config, f, indent=4)

### Feature Engineering

#### Historical Sales Lag and Rolling Features

In [23]:
FEATURE_OUTPUT = PROCESSED_DIR / "daily_origin_features.parquet"

##### Decision:
- PARTITION BY id
   - Each product-store combination is calculated independently.
   - For example, sales history for FOODS_1_001_CA_1 must never be mixed with sales history for FOODS_1_001_TX_1.
- ORDER BY day_num
   - The observations are ordered chronologically within each series
   - Convert d_1, d_2, etc. into integers so DuckDB can order them numerically.
- ROWS BETWEEN 27 PRECEDING AND CURRENT ROW
   - This calculates statistics using the current forecast origin and the previous 27 days.
   - It does not include any future observations.

In [27]:
query = f"""
COPY (
    WITH base AS (
        SELECT
            id,
            item_id,
            dept_id,
            cat_id,
            store_id,
            state_id,

            d,
            CAST(REPLACE(d, 'd_', '') AS INTEGER) AS day_num,

            date,
            wm_yr_wk,
            weekday,
            wday,
            month,
            year,

            event_name_1,
            event_type_1,
            event_name_2,
            event_type_2,

            snap_CA,
            snap_TX,
            snap_WI,

            sell_price,
            sales

        FROM read_parquet('{PRICE_OUTPUT.as_posix()}')
    ),

    features AS (
        SELECT
            *,

            -- Sales observed at the forecast origin
            sales AS sales_at_origin,

            -- Historical sales lags
            LAG(sales, 1) OVER series_window
                AS sales_lag_1,

            LAG(sales, 7) OVER series_window
                AS sales_lag_7,

            LAG(sales, 14) OVER series_window
                AS sales_lag_14,

            LAG(sales, 28) OVER series_window
                AS sales_lag_28,

            -- Seven-day rolling statistics
            AVG(sales) OVER (
                PARTITION BY id
                ORDER BY day_num
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ) AS sales_rolling_mean_7,

            STDDEV_POP(sales) OVER (
                PARTITION BY id
                ORDER BY day_num
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ) AS sales_rolling_std_7,

            -- Twenty-eight-day rolling statistics
            AVG(sales) OVER (
                PARTITION BY id
                ORDER BY day_num
                ROWS BETWEEN 27 PRECEDING AND CURRENT ROW
            ) AS sales_rolling_mean_28,

            STDDEV_POP(sales) OVER (
                PARTITION BY id
                ORDER BY day_num
                ROWS BETWEEN 27 PRECEDING AND CURRENT ROW
            ) AS sales_rolling_std_28,

            MAX(sales) OVER (
                PARTITION BY id
                ORDER BY day_num
                ROWS BETWEEN 27 PRECEDING AND CURRENT ROW
            ) AS sales_rolling_max_28

        FROM base

        WINDOW series_window AS (
            PARTITION BY id
            ORDER BY day_num
        )
    )

    SELECT
        * EXCLUDE (sales)

    FROM features
)
TO '{FEATURE_OUTPUT.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(query)

print("Saved:", FEATURE_OUTPUT)

Saved: c:\Users\cjc72\Desktop\DeliveryLate\AI-Retail-Demand-Intelligence\data\processed\daily_origin_features.parquet


#### Validate

In [28]:
feature_validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(DISTINCT day_num) AS unique_days,

        SUM(
            CASE WHEN sales_at_origin IS NULL THEN 1 ELSE 0 END
        ) AS missing_origin_sales,

        SUM(
            CASE WHEN sales_lag_1 IS NULL THEN 1 ELSE 0 END
        ) AS missing_lag_1,

        SUM(
            CASE WHEN sales_lag_28 IS NULL THEN 1 ELSE 0 END
        ) AS missing_lag_28

    FROM read_parquet('{FEATURE_OUTPUT.as_posix()}')
""").df()

display(feature_validation.T)

,0
total_rows,59181090.0
unique_series,30490.0
unique_days,1941.0
missing_origin_sales,0.0
missing_lag_1,30490.0
missing_lag_28,853720.0


In [29]:
sample_features = con.execute(f"""
    SELECT
        id,
        d,
        day_num,
        sales_at_origin,
        sales_lag_1,
        sales_lag_7,
        sales_lag_14,
        sales_lag_28,
        sales_rolling_mean_7,
        sales_rolling_std_7,
        sales_rolling_mean_28,
        sales_rolling_std_28,
        sales_rolling_max_28

    FROM read_parquet('{FEATURE_OUTPUT.as_posix()}')

    WHERE id = 'FOODS_1_001_CA_1_evaluation'
      AND day_num BETWEEN 25 AND 35

    ORDER BY day_num
""").df()

display(sample_features)

,id,d,day_num,sales_at_origin,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_std_7,sales_rolling_mean_28,sales_rolling_std_28,sales_rolling_max_28
0,FOODS_1_001_CA_1_evaluation,d_25,25,2,0,1,0,<NA>,1.285714,0.880631,1.240000,1.209297,4
1,FOODS_1_001_CA_1_evaluation,d_26,26,2,2,2,0,<NA>,1.285714,0.880631,1.269231,1.194787,4
2,FOODS_1_001_CA_1_evaluation,d_27,27,2,2,0,3,<NA>,1.571429,0.728431,1.296296,1.180546,4
3,FOODS_1_001_CA_1_evaluation,d_28,28,4,2,2,1,<NA>,1.857143,1.124858,1.392857,1.263196,4
4,FOODS_1_001_CA_1_evaluation,d_29,29,2,4,1,3,3,2.000000,1.069045,1.357143,1.230978,4
5,FOODS_1_001_CA_1_evaluation,d_30,30,2,2,2,0,0,2.000000,1.069045,1.428571,1.207967,4
6,FOODS_1_001_CA_1_evaluation,d_31,31,0,2,0,2,0,2.000000,1.069045,1.428571,1.207967,4
7,FOODS_1_001_CA_1_evaluation,d_32,32,2,0,2,1,1,2.000000,1.069045,1.464286,1.209550,4
8,FOODS_1_001_CA_1_evaluation,d_33,33,1,2,2,2,4,1.857143,1.124858,1.357143,1.108870,4
9,FOODS_1_001_CA_1_evaluation,d_34,34,7,1,2,0,2,2.571429,2.128523,1.535714,1.523205,7


### Generate calendar features

In [30]:
CALENDAR_FEATURE_OUTPUT = (
    PROCESSED_DIR / "daily_calendar_features.parquet"
)

query = f"""
COPY (
    SELECT
        *,

        -- Weekend indicator
        CASE
            WHEN weekday IN ('Saturday', 'Sunday') THEN 1
            ELSE 0
        END AS is_weekend,

        -- Day of month
        EXTRACT(DAY FROM date) AS day_of_month,

        -- Week of year
        EXTRACT(WEEK FROM date) AS week_of_year,

        -- Event indicator
        CASE
            WHEN event_name_1 IS NOT NULL
              OR event_name_2 IS NOT NULL
            THEN 1
            ELSE 0
        END AS is_event_day,

        -- State-specific SNAP indicator
        CASE
            WHEN state_id = 'CA' THEN snap_CA
            WHEN state_id = 'TX' THEN snap_TX
            WHEN state_id = 'WI' THEN snap_WI
        END AS is_snap_day

    FROM read_parquet(
        '{PRICE_OUTPUT.as_posix()}'
    )
)
TO '{CALENDAR_FEATURE_OUTPUT.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(query)

print("Saved:", CALENDAR_FEATURE_OUTPUT)

Saved: c:\Users\cjc72\Desktop\DeliveryLate\AI-Retail-Demand-Intelligence\data\processed\daily_calendar_features.parquet


is_snap_day: 
- The dataset contains three SNAP columns, one for each state. However, every product-store series belongs to one particular state.
- Hence this column selects the appropriate indicator for each product-store series.

#### Validate

In [31]:
calendar_feature_validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,

        SUM(CASE WHEN is_weekend NOT IN (0, 1)
            OR is_weekend IS NULL THEN 1 ELSE 0 END
        ) AS invalid_weekend,

        SUM(CASE WHEN is_event_day NOT IN (0, 1)
            OR is_event_day IS NULL THEN 1 ELSE 0 END
        ) AS invalid_event,

        SUM(CASE WHEN is_snap_day NOT IN (0, 1)
            OR is_snap_day IS NULL THEN 1 ELSE 0 END
        ) AS invalid_snap,

        MIN(day_of_month) AS minimum_day_of_month,
        MAX(day_of_month) AS maximum_day_of_month,

        MIN(week_of_year) AS minimum_week,
        MAX(week_of_year) AS maximum_week

    FROM read_parquet(
        '{CALENDAR_FEATURE_OUTPUT.as_posix()}'
    )
""").df()

display(calendar_feature_validation.T)

,0
total_rows,59181090.0
invalid_weekend,0.0
invalid_event,0.0
invalid_snap,0.0
minimum_day_of_month,1.0
maximum_day_of_month,31.0
minimum_week,1.0
maximum_week,53.0


In [32]:
calendar_feature_sample = con.execute(f"""
    SELECT
        id,
        date,
        weekday,
        wday,
        is_weekend,
        day_of_month,
        week_of_year,
        event_name_1,
        is_event_day,
        state_id,
        is_snap_day

    FROM read_parquet(
        '{CALENDAR_FEATURE_OUTPUT.as_posix()}'
    )

    WHERE event_name_1 IS NOT NULL
       OR is_weekend = 1

    ORDER BY date

    LIMIT 20
""").df()

display(calendar_feature_sample)

,id,date,weekday,wday,is_weekend,day_of_month,week_of_year,event_name_1,is_event_day,state_id,is_snap_day
0,HOBBIES_1_389_CA_1_evaluation,2011-01-29,Saturday,1,1,29,4,None,0,CA,0
1,HOBBIES_1_390_CA_1_evaluation,2011-01-29,Saturday,1,1,29,4,None,0,CA,0
2,HOBBIES_1_391_CA_1_evaluation,2011-01-29,Saturday,1,1,29,4,None,0,CA,0
3,HOBBIES_1_392_CA_1_evaluation,2011-01-29,Saturday,1,1,29,4,None,0,CA,0
4,HOUSEHOLD_1_006_CA_1_evaluation,2011-01-29,Saturday,1,1,29,4,None,0,CA,0
5,HOUSEHOLD_1_007_CA_1_evaluation,2011-01-29,Saturday,1,1,29,4,None,0,CA,0
6,HOUSEHOLD_1_008_CA_1_evaluation,2011-01-29,Saturday,1,1,29,4,None,0,CA,0
7,HOUSEHOLD_1_009_CA_1_evaluation,2011-01-29,Saturday,1,1,29,4,None,0,CA,0
8,HOUSEHOLD_1_010_CA_1_evaluation,2011-01-29,Saturday,1,1,29,4,None,0,CA,0
9,HOUSEHOLD_1_011_CA_1_evaluation,2011-01-29,Saturday,1,1,29,4,None,0,CA,0


### Build weekly price features

In [43]:
PRICE_FEATURE_OUTPUT = PROCESSED_DIR / "weekly_price_features.parquet"

query = f"""
COPY (
    WITH raw_prices AS (
        SELECT
            store_id,
            item_id,
            wm_yr_wk,
            sell_price
        FROM read_csv_auto('{PRICE_FILE.as_posix()}')
    ),

    weekly_calendar AS (
        SELECT
            wm_yr_wk,
            ROW_NUMBER() OVER (
                ORDER BY wm_yr_wk
            ) AS week_index
        FROM (
            SELECT DISTINCT wm_yr_wk
            FROM raw_prices
        )
    ),

    item_store_pairs AS (
        SELECT DISTINCT
            store_id,
            item_id
        FROM raw_prices
    ),

    weekly_panel AS (
        SELECT
            p.store_id,
            p.item_id,
            w.wm_yr_wk,
            w.week_index,
            r.sell_price
        FROM item_store_pairs p
        CROSS JOIN weekly_calendar w
        LEFT JOIN raw_prices r
            ON p.store_id = r.store_id
            AND p.item_id = r.item_id
            AND w.wm_yr_wk = r.wm_yr_wk
    ),

    price_history AS (
        SELECT
            *,

            LAG(sell_price) OVER (
                PARTITION BY store_id, item_id
                ORDER BY week_index
            ) AS previous_price,

            AVG(sell_price) OVER (
                PARTITION BY store_id, item_id
                ORDER BY week_index
                ROWS BETWEEN 8 PRECEDING AND 1 PRECEDING
            ) AS previous_8w_avg

        FROM weekly_panel
    )

    SELECT
        store_id,
        item_id,
        wm_yr_wk,
        week_index,
        sell_price,

        CASE
            WHEN sell_price IS NULL THEN 1
            ELSE 0
        END AS price_missing,

        CASE
            WHEN sell_price IS NULL OR previous_price IS NULL
            THEN NULL
            ELSE sell_price - previous_price
        END AS price_change,

        CASE
            WHEN sell_price IS NULL
              OR previous_price IS NULL
              OR previous_price = 0
            THEN NULL
            ELSE (sell_price - previous_price) / previous_price
        END AS price_change_pct,

        CASE
            WHEN sell_price IS NULL
              OR previous_8w_avg IS NULL
              OR previous_8w_avg = 0
            THEN NULL
            ELSE sell_price / previous_8w_avg
        END AS price_vs_8w_avg,

        CASE
            WHEN sell_price IS NULL OR previous_price IS NULL
            THEN NULL
            WHEN sell_price != previous_price THEN 1
            ELSE 0
        END AS price_change_flag

    FROM price_history
)
TO '{PRICE_FEATURE_OUTPUT.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(query)

print(f"Weekly price features saved to: {PRICE_FEATURE_OUTPUT}")

Weekly price features saved to: c:\Users\cjc72\Desktop\DeliveryLate\AI-Retail-Demand-Intelligence\data\processed\weekly_price_features.parquet


#### Validate

In [44]:
price_feature_validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(DISTINCT store_id) AS unique_stores,
        COUNT(DISTINCT item_id) AS unique_items,
        COUNT(DISTINCT wm_yr_wk) AS unique_weeks,

        COUNT(*) FILTER (
            WHERE price_missing = 1
        ) AS missing_price_rows,

        COUNT(*) FILTER (
            WHERE price_missing = 0
        ) AS available_price_rows,

        COUNT(*) FILTER (
            WHERE price_missing NOT IN (0, 1)
               OR price_missing IS NULL
        ) AS invalid_missing_flags,

        COUNT(*) FILTER (
            WHERE price_change_flag NOT IN (0, 1)
              AND price_change_flag IS NOT NULL
        ) AS invalid_change_flags,

        COUNT(*) FILTER (
            WHERE sell_price <= 0
        ) AS non_positive_prices,

        MIN(sell_price) AS minimum_price,
        MAX(sell_price) AS maximum_price,

        MIN(price_vs_8w_avg) AS minimum_price_ratio,
        MAX(price_vs_8w_avg) AS maximum_price_ratio

    FROM read_parquet('{PRICE_FEATURE_OUTPUT.as_posix()}')
""").df()

display(price_feature_validation.T)

,0
total_rows,8.598180e+06
unique_stores,1.000000e+01
unique_items,3.049000e+03
unique_weeks,2.820000e+02
missing_price_rows,1.757059e+06
available_price_rows,6.841121e+06
invalid_missing_flags,0.000000e+00
invalid_change_flags,0.000000e+00
non_positive_prices,0.000000e+00
minimum_price,1.000000e-02


In [45]:
price_feature_duplicate_audit = con.execute(f"""
    SELECT
        COUNT(*) AS duplicate_key_groups
    FROM (
        SELECT
            store_id,
            item_id,
            wm_yr_wk
        FROM read_parquet('{PRICE_FEATURE_OUTPUT.as_posix()}')
        GROUP BY store_id, item_id, wm_yr_wk
        HAVING COUNT(*) > 1
    )
""").df()

display(price_feature_duplicate_audit)

,duplicate_key_groups
0,0


In [46]:
price_feature_sample = con.execute(f"""
    SELECT
        store_id,
        item_id,
        wm_yr_wk,
        week_index,
        sell_price,
        price_missing,
        price_change,
        price_change_pct,
        price_vs_8w_avg,
        price_change_flag
    FROM read_parquet('{PRICE_FEATURE_OUTPUT.as_posix()}')
    WHERE store_id = 'CA_1'
      AND item_id = 'HOBBIES_1_001'
    ORDER BY week_index
    LIMIT 30
""").df()

display(price_feature_sample)

,store_id,item_id,wm_yr_wk,week_index,sell_price,price_missing,price_change,price_change_pct,price_vs_8w_avg,price_change_flag
0,CA_1,HOBBIES_1_001,11101,1,NaN,1,NaN,NaN,NaN,<NA>
1,CA_1,HOBBIES_1_001,11102,2,NaN,1,NaN,NaN,NaN,<NA>
2,CA_1,HOBBIES_1_001,11103,3,NaN,1,NaN,NaN,NaN,<NA>
3,CA_1,HOBBIES_1_001,11104,4,NaN,1,NaN,NaN,NaN,<NA>
4,CA_1,HOBBIES_1_001,11105,5,NaN,1,NaN,NaN,NaN,<NA>
5,CA_1,HOBBIES_1_001,11106,6,NaN,1,NaN,NaN,NaN,<NA>
6,CA_1,HOBBIES_1_001,11107,7,NaN,1,NaN,NaN,NaN,<NA>
7,CA_1,HOBBIES_1_001,11108,8,NaN,1,NaN,NaN,NaN,<NA>
8,CA_1,HOBBIES_1_001,11109,9,NaN,1,NaN,NaN,NaN,<NA>
9,CA_1,HOBBIES_1_001,11110,10,NaN,1,NaN,NaN,NaN,<NA>


In [47]:
price_sample_candidate = con.execute(f"""
    SELECT
        store_id,
        item_id,
        COUNT(sell_price) AS available_price_weeks,
        COUNT(DISTINCT sell_price) AS distinct_prices,
        MIN(wm_yr_wk) AS first_price_week,
        MAX(wm_yr_wk) AS last_price_week
    FROM read_parquet('{PRICE_FEATURE_OUTPUT.as_posix()}')
    GROUP BY store_id, item_id
    HAVING COUNT(sell_price) >= 20
       AND COUNT(DISTINCT sell_price) >= 2
    ORDER BY distinct_prices DESC, available_price_weeks DESC
    LIMIT 10
""").df()

display(price_sample_candidate)

,store_id,item_id,available_price_weeks,distinct_prices,first_price_week,last_price_week
0,TX_3,FOODS_1_095,282,21,11101,11621
1,TX_1,FOODS_1_095,282,21,11101,11621
2,WI_1,FOODS_1_129,282,20,11101,11621
3,WI_2,FOODS_1_129,282,18,11101,11621
4,CA_4,FOODS_1_095,282,18,11101,11621
5,CA_2,FOODS_1_095,282,18,11101,11621
6,WI_3,FOODS_1_129,282,18,11101,11621
7,CA_1,FOODS_1_095,282,18,11101,11621
8,TX_2,FOODS_1_095,282,18,11101,11621
9,TX_3,FOODS_1_096,282,17,11101,11621


In [48]:
sample_store = price_sample_candidate.iloc[0]["store_id"]
sample_item = price_sample_candidate.iloc[0]["item_id"]

price_feature_sample = con.execute(f"""
    SELECT
        store_id,
        item_id,
        wm_yr_wk,
        week_index,
        sell_price,
        price_missing,
        price_change,
        price_change_pct,
        price_vs_8w_avg,
        price_change_flag
    FROM read_parquet('{PRICE_FEATURE_OUTPUT.as_posix()}')
    WHERE store_id = '{sample_store}'
      AND item_id = '{sample_item}'
    ORDER BY week_index
    LIMIT 40
""").df()

display(price_feature_sample)

,store_id,item_id,wm_yr_wk,week_index,sell_price,price_missing,price_change,price_change_pct,price_vs_8w_avg,price_change_flag
0,TX_3,FOODS_1_095,11101,1,3.72,0,NaN,NaN,NaN,<NA>
1,TX_3,FOODS_1_095,11102,2,3.72,0,0.00,0.000000,1.000000,0
2,TX_3,FOODS_1_095,11103,3,3.72,0,0.00,0.000000,1.000000,0
3,TX_3,FOODS_1_095,11104,4,3.72,0,0.00,0.000000,1.000000,0
4,TX_3,FOODS_1_095,11105,5,3.72,0,0.00,0.000000,1.000000,0
5,TX_3,FOODS_1_095,11106,6,3.72,0,0.00,0.000000,1.000000,0
6,TX_3,FOODS_1_095,11107,7,3.72,0,0.00,0.000000,1.000000,0
7,TX_3,FOODS_1_095,11108,8,3.72,0,0.00,0.000000,1.000000,0
8,TX_3,FOODS_1_095,11109,9,3.72,0,0.00,0.000000,1.000000,0
9,TX_3,FOODS_1_095,11110,10,3.72,0,0.00,0.000000,1.000000,0


In [49]:
price_ratio_outliers = con.execute(f"""
    SELECT
        store_id,
        item_id,
        wm_yr_wk,
        week_index,
        sell_price,
        price_change,
        price_change_pct,
        price_vs_8w_avg,
        price_change_flag
    FROM read_parquet('{PRICE_FEATURE_OUTPUT.as_posix()}')
    WHERE price_vs_8w_avg < 0.5
       OR price_vs_8w_avg > 2.0
    ORDER BY
        ABS(LN(price_vs_8w_avg)) DESC
    LIMIT 30
""").df()

display(price_ratio_outliers)

,store_id,item_id,wm_yr_wk,week_index,sell_price,price_change,price_change_pct,price_vs_8w_avg,price_change_flag
0,WI_1,HOBBIES_1_338,11342,146,0.01,-8.97,-0.998886,0.001114,1
1,WI_1,HOBBIES_1_338,11343,147,0.01,0.00,0.000000,0.001272,0
2,WI_1,HOUSEHOLD_1_533,11408,165,0.01,-4.96,-0.997988,0.002012,1
3,WI_3,HOUSEHOLD_1_036,11442,199,0.01,-2.97,-0.996644,0.003356,1
4,WI_3,HOUSEHOLD_1_036,11443,200,0.01,0.00,0.000000,0.003833,0
5,WI_1,FOODS_3_122,11425,182,0.01,-2.49,-0.996000,0.004000,1
6,CA_1,HOUSEHOLD_1_443,11442,199,0.01,-2.46,-0.995951,0.004049,1
7,WI_3,HOUSEHOLD_1_036,11444,201,0.01,0.00,0.000000,0.004469,0
8,WI_2,FOODS_3_242,11538,247,0.05,-9.81,-0.994929,0.005071,1
9,WI_3,HOUSEHOLD_2_182,11608,269,0.05,-6.93,-0.992837,0.007163,1


In [50]:
price_extreme_history = con.execute(f"""
    WITH extreme_prices AS (
        SELECT
            store_id,
            item_id,
            week_index AS extreme_week_index,
            price_vs_8w_avg AS extreme_price_ratio
        FROM read_parquet('{PRICE_FEATURE_OUTPUT.as_posix()}')
        WHERE price_vs_8w_avg < 0.5
           OR price_vs_8w_avg > 2.0
        ORDER BY ABS(LN(price_vs_8w_avg)) DESC
        LIMIT 10
    ),

    raw_prices AS (
        SELECT
            store_id,
            item_id,
            wm_yr_wk,
            sell_price AS raw_sell_price
        FROM read_csv_auto('{PRICE_FILE.as_posix()}')
    )

    SELECT
        e.store_id,
        e.item_id,
        e.extreme_week_index,
        e.extreme_price_ratio,

        f.week_index,
        f.wm_yr_wk,
        f.sell_price,
        r.raw_sell_price,
        f.price_change,
        f.price_change_pct,
        f.price_vs_8w_avg,
        f.price_change_flag

    FROM extreme_prices e
    JOIN read_parquet('{PRICE_FEATURE_OUTPUT.as_posix()}') f
        ON e.store_id = f.store_id
        AND e.item_id = f.item_id
        AND f.week_index BETWEEN
            e.extreme_week_index - 3
            AND e.extreme_week_index + 3

    LEFT JOIN raw_prices r
        ON f.store_id = r.store_id
        AND f.item_id = r.item_id
        AND f.wm_yr_wk = r.wm_yr_wk

    ORDER BY
        e.extreme_price_ratio,
        e.store_id,
        e.item_id,
        f.week_index
""").df()

display(price_extreme_history)

,store_id,item_id,extreme_week_index,extreme_price_ratio,week_index,wm_yr_wk,sell_price,raw_sell_price,price_change,price_change_pct,price_vs_8w_avg,price_change_flag
0,WI_1,HOBBIES_1_338,146,0.001114,143,11339,8.98,8.98,0.00,0.000000,1.000000,0
1,WI_1,HOBBIES_1_338,146,0.001114,144,11340,8.98,8.98,0.00,0.000000,1.000000,0
2,WI_1,HOBBIES_1_338,146,0.001114,145,11341,8.98,8.98,0.00,0.000000,1.000000,0
3,WI_1,HOBBIES_1_338,146,0.001114,146,11342,0.01,0.01,-8.97,-0.998886,0.001114,1
4,WI_1,HOBBIES_1_338,146,0.001114,147,11343,0.01,0.01,0.00,0.000000,0.001272,0
...,...,...,...,...,...,...,...,...,...,...,...,...
65,WI_3,HOUSEHOLD_2_182,269,0.007163,268,11607,6.98,6.98,0.00,0.000000,1.000000,0
66,WI_3,HOUSEHOLD_2_182,269,0.007163,269,11608,0.05,0.05,-6.93,-0.992837,0.007163,1
67,WI_3,HOUSEHOLD_2_182,269,0.007163,270,11609,6.98,6.98,6.93,138.600000,1.141689,1
68,WI_3,HOUSEHOLD_2_182,269,0.007163,271,11610,6.98,6.98,0.00,0.000000,1.141689,0


In [51]:
price_extreme_summary = con.execute(f"""
    SELECT
        COUNT(*) AS total_available_prices,

        COUNT(*) FILTER (
            WHERE sell_price = 0.01
        ) AS prices_equal_001,

        COUNT(*) FILTER (
            WHERE price_vs_8w_avg < 0.5
        ) AS ratios_below_05,

        COUNT(*) FILTER (
            WHERE price_vs_8w_avg > 2.0
        ) AS ratios_above_20,

        COUNT(*) FILTER (
            WHERE price_change_pct < -0.5
        ) AS price_drops_over_50pct,

        COUNT(*) FILTER (
            WHERE price_change_pct > 0.5
        ) AS price_increases_over_50pct

    FROM read_parquet('{PRICE_FEATURE_OUTPUT.as_posix()}')
    WHERE price_missing = 0
""").df()

display(price_extreme_summary.T)

,0
total_available_prices,6841121
prices_equal_001,12
ratios_below_05,2888
ratios_above_20,299
price_drops_over_50pct,2040
price_increases_over_50pct,4627


### Price Feature Findings

- Generated 8,598,180 item-store-week records with no duplicate keys.
- All 6,841,121 available source prices were retained; 1,757,059 records had missing prices.
- **Extreme price movements were verified against the original `sell_prices.csv`, confirming they were present in the source data.**
- Identified 2,888 price-to-8-week-average ratios below 0.5 and 299 above 2.0.
- Retained extreme values without clipping or removal, as they may represent genuine price changes.
- Created `weekly_price_features.parquet` with price changes, percentage changes, trailing 8-week price ratios and price-change indicators.


### Join weekly price features to the daily panel

In [52]:
from pathlib import Path
import duckdb

# Input files
DAILY_FILE = PROCESSED_DIR / "daily_calendar_features.parquet"
PRICE_FEATURE_FILE = PROCESSED_DIR / "weekly_price_features.parquet"

# Output file
DAILY_PRICE_FEATURE_OUTPUT = (
    PROCESSED_DIR / "daily_calendar_price_features.parquet"
)

# Connect to DuckDB
con = duckdb.connect()

# Join weekly price features to the daily panel
con.execute(f"""
    COPY (
        SELECT
            d.*,
            p.week_index,
            p.price_missing,
            p.price_change,
            p.price_change_pct,
            p.price_vs_8w_avg,
            p.price_change_flag

        FROM read_parquet('{DAILY_FILE.as_posix()}') AS d

        LEFT JOIN read_parquet('{PRICE_FEATURE_FILE.as_posix()}') AS p
            ON d.store_id = p.store_id
            AND d.item_id = p.item_id
            AND d.wm_yr_wk = p.wm_yr_wk
    )
    TO '{DAILY_PRICE_FEATURE_OUTPUT.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Daily price feature dataset created successfully.")
print(f"Output: {DAILY_PRICE_FEATURE_OUTPUT}")

Daily price feature dataset created successfully.
Output: c:\Users\cjc72\Desktop\DeliveryLate\AI-Retail-Demand-Intelligence\data\processed\daily_calendar_price_features.parquet


In [53]:
validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(DISTINCT d) AS unique_days,

        COUNT(*) FILTER (
            WHERE price_missing = 1
        ) AS missing_price_rows,

        COUNT(*) FILTER (
            WHERE price_missing = 0
        ) AS available_price_rows,

        COUNT(*) FILTER (
            WHERE week_index IS NULL
        ) AS unmatched_weekly_features,

        COUNT(*) FILTER (
            WHERE price_change IS NULL
        ) AS missing_price_change,

        COUNT(*) FILTER (
            WHERE price_vs_8w_avg IS NULL
        ) AS missing_price_ratio

    FROM read_parquet('{DAILY_PRICE_FEATURE_OUTPUT.as_posix()}')
""").df()

display(validation.T)

,0
total_rows,59181090
unique_series,30490
unique_days,1941
missing_price_rows,12299413
available_price_rows,46881677
unmatched_weekly_features,0
missing_price_change,12512843
missing_price_ratio,12512843


In [54]:
duplicate_check = con.execute(f"""
    SELECT COUNT(*) AS duplicate_groups
    FROM (
        SELECT id, d
        FROM read_parquet('{DAILY_PRICE_FEATURE_OUTPUT.as_posix()}')
        GROUP BY id, d
        HAVING COUNT(*) > 1
    )
""").df()

display(duplicate_check)

,duplicate_groups
0,0


### Join historical sales features

In [60]:
# Output file
FINAL_FEATURE_OUTPUT = (
    PROCESSED_DIR / "daily_model_features.parquet"
)

SALES_FEATURE_FILE = PROCESSED_DIR / "daily_origin_features.parquet"

con.execute(f"""
    COPY (
        SELECT
            d.*,

            s.sales_at_origin,
            s.sales_lag_1,
            s.sales_lag_7,
            s.sales_lag_14,
            s.sales_lag_28,

            s.sales_rolling_mean_7,
            s.sales_rolling_std_7,
            s.sales_rolling_mean_28,
            s.sales_rolling_std_28,
            s.sales_rolling_max_28

        FROM read_parquet(
            '{DAILY_PRICE_FEATURE_OUTPUT.as_posix()}'
        ) AS d

        LEFT JOIN read_parquet(
            '{SALES_FEATURE_FILE.as_posix()}'
        ) AS s

            ON d.id = s.id
            AND d.d = s.d

    )
    TO '{FINAL_FEATURE_OUTPUT.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Final feature dataset created successfully.")
print(f"Output: {FINAL_FEATURE_OUTPUT}")

Final feature dataset created successfully.
Output: c:\Users\cjc72\Desktop\DeliveryLate\AI-Retail-Demand-Intelligence\data\processed\daily_model_features.parquet


In [61]:
final_validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_series,
        COUNT(DISTINCT d) AS unique_days,

        COUNT(*) - COUNT(DISTINCT (id, d))
            AS duplicate_rows,

        COUNT(*) FILTER (
            WHERE sales_at_origin IS NULL
        ) AS missing_sales_at_origin,

        COUNT(*) FILTER (
            WHERE sales_lag_1 IS NULL
        ) AS missing_lag_1,

        COUNT(*) FILTER (
            WHERE sales_rolling_mean_7 IS NULL
        ) AS missing_rolling_mean_7,

        COUNT(*) FILTER (
            WHERE sales_rolling_mean_28 IS NULL
        ) AS missing_rolling_mean_28

    FROM read_parquet('{FINAL_FEATURE_OUTPUT.as_posix()}')
""").df()

display(final_validation.T)

,0
total_rows,59181090
unique_series,30490
unique_days,1941
duplicate_rows,0
missing_sales_at_origin,0
missing_lag_1,30490
missing_rolling_mean_7,0
missing_rolling_mean_28,0


### Final Feature Dataset Audit

In [62]:
final_schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{FINAL_FEATURE_OUTPUT.as_posix()}')
""").df()

display(final_schema)

,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,item_id,VARCHAR,YES,None,None,None
2,dept_id,VARCHAR,YES,None,None,None
3,cat_id,VARCHAR,YES,None,None,None
4,store_id,VARCHAR,YES,None,None,None
5,state_id,VARCHAR,YES,None,None,None
6,d,VARCHAR,YES,None,None,None
7,sales,BIGINT,YES,None,None,None
8,date,DATE,YES,None,None,None
9,wm_yr_wk,BIGINT,YES,None,None,None


### Consolidated Feature Dataset Findings

- Integrated historical sales, calendar and weekly price features into one dataset.
- Preserved all 59,181,090 observations across 30,490 series and 1,941 days.
- Verified zero duplicate `(id, d)` records.
- Confirmed all sales-at-origin values are available.
- Observed 30,490 missing lag-1 values, corresponding to the first observation of each series.
- Rolling statistics were successfully generated without missing values.

**Conclusion:** The consolidated daily feature dataset passed structural validation and is ready for supervised training-example preparation.